# Nombre: Byron Ortiz

# Workshop: Building an Information Retrieval System for Podcast Episodes

## Objective:
Create an Information Retrieval (IR) system that processes a dataset of podcast transcripts and, given a query, returns the episodes where the host and guest discuss the query topic. Use TF-IDF and BERT for vector space representation and compare the results.

Instructions:

### Step 1: Import Libraries
Import necessary libraries for data handling, text processing, and machine learning.

Se importa las siguientes librerias

In [1]:
import numpy as np
import pandas as pd
import nltk
import string
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from transformers import BertTokenizer, TFBertModel
from sklearn.feature_extraction.text import TfidfVectorizer
import time

### Step 2: Load the Dataset

Load the dataset of podcast transcripts.

Primero se carga el dataset de los podcast

In [2]:
podcast_df = pd.read_csv('datos/podcastdata_dataset.csv')
print(podcast_df)

      id             guest                                              title  \
0      1       Max Tegmark                                           Life 3.0   
1      2     Christof Koch                                      Consciousness   
2      3     Steven Pinker                            AI in the Age of Reason   
3      4     Yoshua Bengio                                      Deep Learning   
4      5   Vladimir Vapnik                               Statistical Learning   
..   ...               ...                                                ...   
314  321      Ray Kurzweil    Singularity, Superintelligence, and Immortality   
315  322  Rana el Kaliouby   Emotion AI, Social Robots, and Self-Driving Cars   
316  323        Will Sasso  Comedy, MADtv, AI, Friendship, Madness, and Pr...   
317  324   Daniel Negreanu                                              Poker   
318  325     Michael Levin  Biology, Life, Aliens, Evolution, Embryogenesi...   

                           

### Step 3: Text Preprocessing
* Delete punctuation
* Delete stop words

obtenemos el corpus de la columna text

In [3]:
corpus = podcast_df['text']
print(corpus)

0      As part of MIT course 6S099, Artificial Genera...
1      As part of MIT course 6S099 on artificial gene...
2      You've studied the human mind, cognition, lang...
3      What difference between biological neural netw...
4      The following is a conversation with Vladimir ...
                             ...                        
314    By the time he gets to 2045, we'll be able to ...
315    there's a broader question here, right? As we ...
316    Once this whole thing falls apart and we are c...
317    you could be the seventh best player in the wh...
318    turns out that if you train a planarian and th...
Name: text, Length: 319, dtype: object


TF-IDF Text Preprocessing

Se elimina la puntuacion y se transforma todo a minusculas

In [6]:
corpus_procesado = []
for doc in corpus: 
    corpus_procesado.append(doc.lower().translate(str.maketrans('', '', string.punctuation)))
print('corpus: ' , pd.DataFrame(corpus_procesado).head())
print('tamaño del corpus: ', len(corpus_procesado)) 

corpus:                                                     0
0  as part of mit course 6s099 artificial general...
1  as part of mit course 6s099 on artificial gene...
2  youve studied the human mind cognition languag...
3  what difference between biological neural netw...
4  the following is a conversation with vladimir ...
tamaño del corpus:  319


se agrega al dataframe el corpus sin puntuacion

In [7]:
podcast_df['text_nopunct']=corpus_procesado
print(podcast_df.head())

   id            guest                    title  \
0   1      Max Tegmark                 Life 3.0   
1   2    Christof Koch            Consciousness   
2   3    Steven Pinker  AI in the Age of Reason   
3   4    Yoshua Bengio            Deep Learning   
4   5  Vladimir Vapnik     Statistical Learning   

                                                text  \
0  As part of MIT course 6S099, Artificial Genera...   
1  As part of MIT course 6S099 on artificial gene...   
2  You've studied the human mind, cognition, lang...   
3  What difference between biological neural netw...   
4  The following is a conversation with Vladimir ...   

                                        text_nopunct  
0  as part of mit course 6s099 artificial general...  
1  as part of mit course 6s099 on artificial gene...  
2  youve studied the human mind cognition languag...  
3  what difference between biological neural netw...  
4  the following is a conversation with vladimir ...  


 usando la libreira nltk importamos las stopwords

In [8]:
# Descargar el recurso stopwords
nltk.download('stopwords')

# Cargar las stopwords
stopw = set(stopwords.words('english'))
len(stopw)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\byron\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


179

Se elimina las stopwords del corpus procesado anteriormente

In [9]:
corpus_sinstw=[]
for doc in corpus_procesado:
    clean_doc = []
    doc_array = doc.split(' ')
    for word in doc_array:
        if word not in stopw:
           clean_doc.append(word)
    corpus_sinstw.append(' '.join(clean_doc))
print('longitud del corpus sin stopwords: ', len(corpus_sinstw))
print('corpus sin stopwords: ',pd.DataFrame(corpus_sinstw).head())

longitud del corpus sin stopwords:  319
corpus sin stopwords:                                                     0
0  part mit course 6s099 artificial general intel...
1  part mit course 6s099 artificial general intel...
2  youve studied human mind cognition language vi...
3  difference biological neural networks artifici...
4  following conversation vladimir vapnik hes co ...


se agrega al dataframe el corpus sin stop words

In [10]:
podcast_df['text_nostopw']= corpus_sinstw
print(podcast_df)

      id             guest                                              title  \
0      1       Max Tegmark                                           Life 3.0   
1      2     Christof Koch                                      Consciousness   
2      3     Steven Pinker                            AI in the Age of Reason   
3      4     Yoshua Bengio                                      Deep Learning   
4      5   Vladimir Vapnik                               Statistical Learning   
..   ...               ...                                                ...   
314  321      Ray Kurzweil    Singularity, Superintelligence, and Immortality   
315  322  Rana el Kaliouby   Emotion AI, Social Robots, and Self-Driving Cars   
316  323        Will Sasso  Comedy, MADtv, AI, Friendship, Madness, and Pr...   
317  324   Daniel Negreanu                                              Poker   
318  325     Michael Levin  Biology, Life, Aliens, Evolution, Embryogenesi...   

                           

Bert Text Preprocessing

Se carga el modelo preentrenado Bert y su tokenizer

In [11]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = TFBertModel.from_pretrained('bert-base-uncased')

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

###  Step 4: Vector Space Representation - TF-IDF

Create TF-IDF vector representations of the transcripts.

Se crea la matriz tfidf para los textos ya preprocesados

In [12]:
vectorizer = TfidfVectorizer()
matriz_tfidf = vectorizer.fit_transform(podcast_df['text_nostopw'])
print('tamaño de la matriz tfidf: ', matriz_tfidf.shape)

tamaño de la matriz tfidf:  (319, 49728)


Se vectoriza la consulta hecha con el mismo vectorizer

In [13]:
query = 'Computer Science' 
vector_consulta = vectorizer.transform([query])
print('tamaño de la query vector: ', vector_consulta.shape)

tamaño de la query vector:  (1, 49728)


Se realiza la similitud coseno entre la matriz_tfidf y el vector_consulta

In [14]:
similitud_tfidf = cosine_similarity(matriz_tfidf,vector_consulta)
print('similitud coseno usando tfidf: ', pd.DataFrame(similitud_tfidf).head())
print('tamaño : ', similitud_tfidf.shape)

similitud coseno usando tfidf:            0
0  0.045080
1  0.072728
2  0.014514
3  0.056815
4  0.023408
tamaño :  (319, 1)


### Step 5: Vector Space Representation - BERT

Create BERT vector representations of the transcripts using a pre-trained BERT model.

Se crea una función para generar los embeddings de BERT a partir de los textos

In [15]:
def generar_embeddings_bert(textos):
    inicio = time.time()
    embeddings = []
    for texto in textos:
        entradas = tokenizer(texto, return_tensors='tf', padding=True, truncation=True)
        salidas = model(**entradas)
        embeddings.append(salidas.last_hidden_state[:, 0, :])  # Usar representación del token [CLS]
    fin = time.time()
    print(f'Tiempo total de procesamiento: {fin - inicio:.2f} segundos')
    return np.array(embeddings).transpose(0, 2, 1)

Se generan los embeddings usando BERT para el corpus

In [16]:
embeddings_bert = generar_embeddings_bert(corpus)

Tiempo total de procesamiento: 472.56 segundos


In [17]:
print("Bert Embeddings:", embeddings_bert)
print("Bert tamaño:", embeddings_bert.shape)

Bert Embeddings: [[[-0.13343129]
  [-0.20641235]
  [ 0.00869849]
  ...
  [ 0.16827932]
  [ 0.4736129 ]
  [ 0.47503072]]

 [[ 0.27045074]
  [-0.00788647]
  [ 0.00813641]
  ...
  [-0.0830892 ]
  [ 0.7754291 ]
  [ 0.32223815]]

 [[ 0.4752612 ]
  [-0.01439897]
  [-0.37041193]
  ...
  [-0.08524533]
  [ 0.49683863]
  [ 0.369994  ]]

 ...

 [[ 0.18494345]
  [-0.43932334]
  [ 0.11626427]
  ...
  [ 0.11742233]
  [ 0.72239137]
  [ 0.3660169 ]]

 [[-0.01205373]
  [-0.18836772]
  [-0.06401554]
  ...
  [-0.18816544]
  [ 0.6607347 ]
  [ 0.68429625]]

 [[-0.15623447]
  [-0.33060414]
  [-0.1911866 ]
  ...
  [ 0.06854291]
  [ 0.70506334]
  [ 0.37314472]]]
Bert tamaño: (319, 768, 1)


Se genera el embedding para la consulta usando bert

In [20]:
query=['Computer Science']
query_bert= generar_embeddings_bert(query)
print('query bert: ', pd.DataFrame(query_bert.reshape(1,768)).head())
print('tamaño query bert: ', query_bert.shape)

Tiempo total de procesamiento: 0.27 segundos
query bert:          0         1         2         3         4         5         6    \
0  0.171412  0.482464 -0.781255  0.096592 -0.329068  0.258919  0.259098   

        7         8         9    ...       758       759       760       761  \
0  1.108088 -0.122094 -0.154697  ...  0.334298 -0.223618  0.440867  0.364894   

       762       763       764       765       766       767  
0 -0.49323  0.013611 -0.258381 -0.448226  0.323409  0.727026  

[1 rows x 768 columns]
tamaño query bert:  (1, 768, 1)


### Step 6: Query Processing

Define a function to process the query and compute similarity scores using both TF-IDF and BERT embeddings.

# TFIDF

Se crea una funcion para  obtener la similitud coseno usando tfidf con el parametro enviado que seria la query

In [21]:
def retrieve_tfidf(query):
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(matriz_tfidf,query_vector)
    similarities_df =pd.DataFrame(similarities, columns=['sim'])
    similarities_df['ep'] = podcast_df['title']
    return similarities_df

In [22]:
tfidf_similarity_consulta=retrieve_tfidf('ia')
tfidf_similarity_consulta

,sim,ep
0,0.0,Life 3.0
1,0.0,Consciousness
2,0.0,AI in the Age of Reason
3,0.0,Deep Learning
4,0.0,Statistical Learning
...,...,...
314,0.0,"Singularity, Superintelligence, and Immortality"
315,0.0,"Emotion AI, Social Robots, and Self-Driving Cars"
316,0.0,"Comedy, MADtv, AI, Friendship, Madness, and Pr..."
317,0.0,Poker


# Bert

Se crea una funcion para  obtener la similitud coseno usando bert con el parametro enviado que seria la query

In [25]:
def retrieve_bert(query):
    query_bert =  generar_embeddings_bert(query)
    similarities = cosine_similarity(embeddings_bert.reshape(319,768),query_bert.reshape(1,768))
    similarities_df =pd.DataFrame(similarities, columns=['sim'])
    similarities_df['ep'] = podcast_df['title']
    return similarities_df

In [26]:
bert_similarity_consulta=retrieve_bert(['ia'])
bert_similarity_consulta

Tiempo total de procesamiento: 0.29 segundos


,sim,ep
0,0.627895,Life 3.0
1,0.649277,Consciousness
2,0.635224,AI in the Age of Reason
3,0.504697,Deep Learning
4,0.623663,Statistical Learning
...,...,...
314,0.560683,"Singularity, Superintelligence, and Immortality"
315,0.582955,"Emotion AI, Social Robots, and Self-Driving Cars"
316,0.638889,"Comedy, MADtv, AI, Friendship, Madness, and Pr..."
317,0.623576,Poker


# Resultados

TFIDF

Se ordena por similitud coseno para mostrar los 15 episodios mas similares a la consulta usando TFIDF

In [27]:
idf_result_df = retrieve_tfidf('ia')
result_idf = idf_result_df.sort_values(by='sim', ascending=False)
print('los 15 episodios más similares usando tfidf: \n', result_idf.head(15))

los 15 episodios más similares usando tfidf: 
           sim                                                 ep
65   0.101752      Thinking Fast and Slow, Deep Learning, and AI
136  0.016044  Supernovae, Dark Energy, Aliens & the Expandin...
0    0.000000                                           Life 3.0
219  0.000000  Cyc and the Quest to Solve Common Sense Reason...
217  0.000000  Programming, Algorithms, Hard Problems & the G...
216  0.000000  Virtual Reality, Social Media & the Future of ...
215  0.000000                                           Robotics
214  0.000000                               Viruses and Vaccines
213  0.000000  OpenAI Codex, GPT-3, Robotics, and the Future ...
212  0.000000         Isaac Newton and the Philosophy of Science
211  0.000000  Gravitational Waves and the Most Precise Devic...
210  0.000000       Nature of Reality, Dreams, and Consciousness
209  0.000000                 The Secret History of Psychedelics
208  0.000000                              

BERT

Se ordena por similitud coseno para mostrar los 15 episodios mas similares a la consulta usando BERT

In [28]:
bert_result_df = retrieve_bert(['ia'])
result_bert = bert_result_df.sort_values(by='sim', ascending=False)
print('los 15 episodios más similares usando bert: \n', result_bert.head(15))

Tiempo total de procesamiento: 0.26 segundos
los 15 episodios más similares usando bert: 
           sim                                                 ep
16   0.724326  Revolutionary Ideas in Science, Math, and Society
199  0.713550                        Totalitarianism and Anarchy
133  0.708087  On the Nature of Good and Evil, Genius and Mad...
273  0.706432        Bitcoin, Inflation, and the Future of Money
287  0.694157                             iPhone, iPod, and Nest
129  0.693381         Computational Complexity and Consciousness
155  0.692062    Comedy, Power, Conspiracy Theories, and Freedom
164  0.689464  Philosophy of Violence, Power, and the Martial...
210  0.689075       Nature of Reality, Dreams, and Consciousness
161  0.688338  The Future of Computing, AI, Life, and Conscio...
156  0.688145  Rocket Engines and Electric Spacecraft Propulsion
157  0.686155   The Next Generation of Big Ideas and Brave Minds
165  0.685319  Deep Work, Focus, Productivity, Email, and Soc...